In [1]:
!pip install gradio

In [2]:
!pip install --upgrade gradio

In [ ]:
import json
from pathlib import Path
import gradio as gr

def convert_ipynb_to_md(file_obj) -> str:
    if file_obj is None:
        return "กรุณาลากไฟล์มาวางก่อนครับ"
    
    file_path = Path(file_obj.name)
    
    if file_path.suffix != ".ipynb":
        return "ERROR: กรุณาอัปโหลดไฟล์ที่มีนามสกุล .ipynb เท่านั้น"

    try:
        with open(file_path, "r", encoding="utf-8") as f:
            notebook = json.load(f)
    except Exception as e:
        return f"เกิดข้อผิดพลาดในการอ่านไฟล์: {str(e)}"

    md_lines = [f"# Notebook Context: {file_path.name}\n"]

    for idx, cell in enumerate(notebook.get("cells", []), 1):
        cell_type = cell.get("cell_type")
        source = "".join(cell.get("source", []))

        if not source.strip():
            continue

        if cell_type == "markdown":
            md_lines.append(f"\n{source}\n")

        elif cell_type == "code":
            md_lines.append(f"\n```python\n{source}\n```\n")

            outputs = cell.get("outputs", [])
            output_text = []
            for out in outputs:
                if out.get("output_type") == "stream":
                    output_text.append("".join(out.get("text", [])))
                elif out.get("output_type") in ["execute_result", "display_data"]:
                    data = out.get("data", {})
                    if "text/plain" in data:
                        output_text.append("".join(data["text/plain"]))

            if output_text:
                full_output = "".join(output_text).strip()
                md_lines.append(f"<details><summary>Output Cell {idx}</summary>\n\n```text\n{full_output}\n```\n</details>\n")

    full_md = "\n".join(md_lines)
    
    # ---------------------------------------------------------
    # กำหนด Path ไปยังโฟลเดอร์ Downloads ของผู้ใช้
    # ---------------------------------------------------------
    downloads_dir = Path.home() / "Downloads"
    
    # ดึงชื่อไฟล์ดั้งเดิม (Original Name) แล้วเปลี่ยนนามสกุลเป็น .md
    original_filename = getattr(file_obj, "orig_name", file_path.name)
    output_filename = Path(original_filename).stem + ".md"
    output_md_path = downloads_dir / output_filename
    
    # บันทึกไฟล์ลงโฟลเดอร์ Downloads
    with open(output_md_path, "w", encoding="utf-8") as f:
        f.write(full_md)

    return f"แปลงไฟล์สำเร็จแล้ว!\nบันทึกไว้ที่ Downloads: {output_md_path}\n\n--- คัดลอกข้อความด้านล่างไปใช้เป็น Prompt ได้เลย ---\n\n" + full_md


with gr.Blocks(title="IPYNB to MD Converter") as demo:
    gr.Markdown("## 📄 IPYNB to Markdown Prompt Converter\nลากไฟล์ `.ipynb` มาวางในช่องด้านล่างเพื่อแปลงเป็น Markdown สำหรับส่งให้ AI")
    
    with gr.Row():
        file_input = gr.File(label="ลากไฟล์ .ipynb มาวางที่นี่", file_types=[".ipynb"])
    
    btn = gr.Button("แปลงไฟล์ (Convert)", variant="primary")
    output_text = gr.Textbox(label="ผลลัพธ์ Markdown", lines=15)
    
    btn.click(fn=convert_ipynb_to_md, inputs=file_input, outputs=output_text)

if __name__ == "__main__":
    demo.launch()